# 从零实现 U-Net：像素级分割、奇偶尺寸对齐与工程评估

这份 notebook 只使用 PyTorch 基础层，手写 `DoubleConv`、`Down`、`Up` 和 `UNet` 的 `forward`，完整走通二值语义分割：像素 logits、BCEWithLogits + Dice、空 mask 指标策略、奇数尺寸 skip 对齐、合成几何图形小样本过拟合、validation 阈值冻结、重叠 tile 推理和权重指纹。

模型在 CPU 上训练极小合成数据；它用于验证实现和评估合同，不代表医学、遥感、文档或自动驾驶分割效果。


## 1. 分割系统的 shape 主线

```text
image [N,C,H,W]
  -> 编码器：DoubleConv + Down，增加语义、降低分辨率
  -> bottleneck
  -> 解码器：上采样 + 对齐 skip + concat + DoubleConv
  -> pixel logits [N,1,H,W]
  -> sigmoid 仅用于概率/阈值
  -> validation 选择阈值 -> 冻结 -> test
```

二值分割最后一层输出一个通道；多类互斥分割通常输出 `C` 通道并配合 cross-entropy。不要把类别分类的 `[N,C]` logits 和像素级 `[N,C,H,W]` logits 混为一谈。


In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)

from hashlib import sha256
import json
import math
import random
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

SEED = 230728
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.set_num_threads(1)
DEVICE = torch.device("cpu")

assert DEVICE.type == "cpu"
assert torch.get_num_threads() == 1
print({"torch": torch.__version__, "device": str(DEVICE), "seed": SEED})


## 2. 为什么奇数尺寸会让 skip connection 对不上

`MaxPool2d(2)` 对奇数长度执行向下取整，而 `ConvTranspose2d(..., stride=2)` 通常精确翻倍。例如 31 经池化变 15，再上采样回 30，不会自动回到 31。U-Net 要把解码特征与同层编码特征在通道维拼接，空间维必须先对齐。

下面的 `align_to` 采用中心裁剪/对称 padding：source 太大先裁，太小再补。生产必须固定 padding mode 和左右分配规则，否则训练、导出与服务端可能相差一像素。


In [ ]:
def align_to(source, reference):
    if source.ndim != 4 or reference.ndim != 4:
        raise ValueError("align_to 只接受 NCHW 张量")
    target_h, target_w = reference.shape[-2:]
    height, width = source.shape[-2:]

    if height > target_h:
        top = (height - target_h) // 2
        source = source[..., top:top + target_h, :]
    if width > target_w:
        left = (width - target_w) // 2
        source = source[..., :, left:left + target_w]

    pad_h = target_h - source.shape[-2]
    pad_w = target_w - source.shape[-1]
    if pad_h < 0 or pad_w < 0:
        raise RuntimeError("裁剪后仍大于目标尺寸")
    source = F.pad(source, [pad_w // 2, pad_w - pad_w // 2,
                            pad_h // 2, pad_h - pad_h // 2])
    return source

small = torch.ones(1, 2, 4, 6)
large = torch.ones(1, 2, 7, 9)
assert align_to(small, large).shape == large.shape
assert align_to(large, small).shape == small.shape
assert float(align_to(small, large).sum()) == float(small.sum())


## 3. 编码器：DoubleConv 与 Down

原始 U-Net 每一级使用两个 3×3 VALID 卷积，所以 skip 需要中心裁剪；很多现代实现改用 `padding=1` 的 SAME 卷积。这里采用现代 SAME 变体，但 pooling 遇到奇数尺寸仍需显式对齐。

`DoubleConv` 不改变空间尺寸，`Down` 先做 2×2 最大池化再提取特征。卷积后不用 sigmoid，让中间表示保留无界数值范围。


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.ReLU(inplace=False),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.ReLU(inplace=False),
        )

    def forward(self, x):
        return self.net(x)

class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_channels, out_channels))

    def forward(self, x):
        return self.net(x)

encoder_probe = Down(4, 8)(torch.zeros(2, 4, 31, 35))
assert encoder_probe.shape == (2, 8, 15, 17)
assert sum(isinstance(m, nn.Conv2d) for m in DoubleConv(1, 4).modules()) == 2


## 4. 解码器：上采样、skip concat 与逐像素 logits

`Up` 先用转置卷积扩大 decoder feature，再对齐到 encoder skip 的空间尺寸，沿 channel 维 `dim=1` 拼接，最后用 `DoubleConv` 融合。拼接保留编码器的高分辨率定位信息；相加则要求通道一致且会混合两路语义，不是同一操作。

最后的 1×1 卷积把每个像素的 feature 映射为一个 logit。`forward` 不做 sigmoid，因为 `BCEWithLogitsLoss` 将 sigmoid 与交叉熵组合，数值更稳定。


In [ ]:
class Up(nn.Module):
    def __init__(self, decoder_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(decoder_channels, out_channels, kernel_size=2, stride=2)
        self.fuse = DoubleConv(out_channels + skip_channels, out_channels)

    def forward(self, decoder_feature, skip_feature):
        decoder_feature = align_to(self.up(decoder_feature), skip_feature)
        merged = torch.cat([skip_feature, decoder_feature], dim=1)
        return self.fuse(merged)

class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, base=4):
        super().__init__()
        self.inc = DoubleConv(in_channels, base)
        self.down1 = Down(base, base * 2)
        self.down2 = Down(base * 2, base * 4)
        self.up1 = Up(base * 4, base * 2, base * 2)
        self.up2 = Up(base * 2, base, base)
        self.out_conv = nn.Conv2d(base, out_channels, kernel_size=1)

    def forward(self, x):
        if x.ndim != 4:
            raise ValueError("UNet 期望 [N,C,H,W]")
        x1 = self.inc(x)
        x2 = self.down1(x1)
        bottleneck = self.down2(x2)
        decoded = self.up1(bottleneck, x2)
        decoded = self.up2(decoded, x1)
        logits = self.out_conv(decoded)
        if logits.shape[-2:] != x.shape[-2:]:
            raise RuntimeError("输出与输入空间尺寸不一致")
        return logits

unet = UNet().to(DEVICE)
odd_logits = unet(torch.zeros(2, 1, 31, 35))
assert odd_logits.shape == (2, 1, 31, 35)
assert odd_logits.dtype.is_floating_point
assert odd_logits.requires_grad
assert torch.isfinite(odd_logits).all()
assert not any(isinstance(module, nn.Sigmoid) for module in unet.modules())
assert sum(p.numel() for p in unet.parameters()) < 20_000


## 5. BCEWithLogits + soft Dice

像素 BCE 对每个位置提供稳定梯度，但前景极少时容易被大量背景主导。soft Dice 直接优化区域重叠：

$$\mathrm{Dice}=\frac{2\sum_i p_i y_i+\epsilon}{\sum_i p_i+\sum_i y_i+\epsilon}.$$

组合损失 $L=\alpha L_{BCE}+(1-\alpha)(1-\mathrm{Dice})$ 同时保留像素概率项与区域项。这里逐样本计算再求均值，避免大目标完全压过小目标；`eps` 还定义了空预测/空真值时的数值行为。


In [ ]:
def soft_dice_loss(logits, targets, eps=1e-6):
    if logits.shape != targets.shape:
        raise ValueError(f"logits/targets shape 不同: {logits.shape} vs {targets.shape}")
    probabilities = torch.sigmoid(logits)
    dims = tuple(range(1, logits.ndim))
    intersection = (probabilities * targets).sum(dim=dims)
    denominator = probabilities.sum(dim=dims) + targets.sum(dim=dims)
    dice = (2 * intersection + eps) / (denominator + eps)
    return 1 - dice.mean()

def segmentation_loss(logits, targets, bce_weight=0.5):
    bce = F.binary_cross_entropy_with_logits(logits, targets)
    dice = soft_dice_loss(logits, targets)
    return bce_weight * bce + (1 - bce_weight) * dice

perfect_logits = torch.tensor([[[[12.0, -12.0], [-12.0, 12.0]]]])
perfect_target = torch.tensor([[[[1.0, 0.0], [0.0, 1.0]]]])
wrong_logits = -perfect_logits
assert segmentation_loss(perfect_logits, perfect_target) < 0.01
assert segmentation_loss(wrong_logits, perfect_target) > 5.0
assert torch.isfinite(segmentation_loss(torch.zeros_like(perfect_logits), perfect_target))


## 6. IoU/Dice 与空 mask 必须先约定

硬预测下 $IoU=|P\cap G|/|P\cup G|$，$Dice=2|P\cap G|/(|P|+|G|)$。当预测和真值都为空，分母为零：

- `perfect`：记 1，适合“正确判空也有价值”的任务；
- `skip`：该样本不进入重叠均值，但应另报空样本识别率；
- 若真值空但出现假阳性，IoU/Dice 仍为 0，不能跳过。

报告必须写明 policy、逐样本还是全局聚合、阈值和 resize 后处理。


In [ ]:
def binary_overlap_metrics(probabilities, targets, threshold=0.5, empty_policy="perfect"):
    if probabilities.shape != targets.shape:
        raise ValueError("probabilities/targets shape 不一致")
    if empty_policy not in {"perfect", "skip"}:
        raise ValueError("empty_policy 必须是 perfect 或 skip")
    predictions = probabilities >= threshold
    truth = targets >= 0.5
    ious, dices, empty_correct = [], [], 0
    for prediction, target in zip(predictions, truth):
        intersection = int((prediction & target).sum())
        union = int((prediction | target).sum())
        total = int(prediction.sum() + target.sum())
        both_empty = union == 0
        if both_empty:
            empty_correct += 1
            if empty_policy == "skip":
                continue
        ious.append(1.0 if both_empty else intersection / union)
        dices.append(1.0 if both_empty else 2 * intersection / total)
    return {"iou": float(np.mean(ious)) if ious else math.nan,
            "dice": float(np.mean(dices)) if dices else math.nan,
            "evaluated": len(ious), "empty_correct": empty_correct}

empty_prob = torch.zeros(1, 1, 3, 3)
empty_true = torch.zeros_like(empty_prob)
false_positive = empty_prob.clone(); false_positive[..., 1, 1] = 1.0
assert binary_overlap_metrics(empty_prob, empty_true)["iou"] == 1.0
assert binary_overlap_metrics(empty_prob, empty_true, empty_policy="skip")["evaluated"] == 0
assert binary_overlap_metrics(false_positive, empty_true)["dice"] == 0.0


## 7. 合成几何分割数据与独立随机流

每张图随机放置圆或矩形，mask 是精确真值；部分样本故意为空。输入由前景亮度、背景梯度和噪声组成。训练、validation、test 使用不同 seed，验证集只选阈值，测试集只在阈值冻结后报告。

这类前景和背景几乎线性可分，比真实边界、遮挡、弱标注和域偏移简单得多。它适合单元测试，不适合证明模型“会分割真实目标”。


In [ ]:
def make_segmentation_data(count, height, width, seed):
    generator = torch.Generator().manual_seed(seed)
    yy, xx = torch.meshgrid(torch.arange(height), torch.arange(width), indexing="ij")
    images, masks = [], []
    for index in range(count):
        mask = torch.zeros(height, width, dtype=torch.float32)
        if index % 7 != 0:
            center_y = int(torch.randint(8, height - 7, (1,), generator=generator))
            center_x = int(torch.randint(8, width - 7, (1,), generator=generator))
            if index % 2 == 0:
                radius = int(torch.randint(4, 7, (1,), generator=generator))
                mask[((yy - center_y) ** 2 + (xx - center_x) ** 2) <= radius ** 2] = 1
            else:
                half_h = int(torch.randint(3, 7, (1,), generator=generator))
                half_w = int(torch.randint(3, 7, (1,), generator=generator))
                mask[max(0, center_y-half_h):min(height, center_y+half_h),
                     max(0, center_x-half_w):min(width, center_x+half_w)] = 1
        gradient = torch.linspace(0, 0.12, width).repeat(height, 1)
        noise = 0.05 * torch.randn((height, width), generator=generator)
        image = (0.12 + gradient + 0.72 * mask + noise).clamp(0, 1)
        images.append(image.unsqueeze(0)); masks.append(mask.unsqueeze(0))
    return torch.stack(images), torch.stack(masks)

train_x23, train_y23 = make_segmentation_data(21, 31, 35, SEED + 1)
val_x23, val_y23 = make_segmentation_data(7, 31, 35, SEED + 2)
test_x23, test_y23 = make_segmentation_data(7, 31, 35, SEED + 3)
assert train_x23.shape == train_y23.shape == (21, 1, 31, 35)
assert int((train_y23.flatten(1).sum(1) == 0).sum()) == 3
assert not torch.equal(train_x23[:7], val_x23)
assert set(torch.unique(train_y23).tolist()) == {0.0, 1.0}


## 8. 小样本过拟合：检查 forward、loss 与 optimizer

先固定初始化，再让网络反复看 21 张图。Adam 是为了在短 notebook 中快速验证链路；真实训练应在训练集内比较优化器、学习率、权重衰减和 scheduler，并记录像素采样与增强。

过拟合验收看 loss 显著下降、梯度有限非零、训练 Dice 很高。validation 结果不作为“泛化证明”。


In [ ]:
torch.manual_seed(SEED + 10)
model23 = UNet(base=4).to(DEVICE)
optimizer23 = torch.optim.Adam(model23.parameters(), lr=0.015)

model23.train()
with torch.no_grad():
    initial_loss23 = float(segmentation_loss(model23(train_x23), train_y23))
loss_curve23 = []
for step in range(100):
    optimizer23.zero_grad(set_to_none=True)
    logits23 = model23(train_x23)
    loss23 = segmentation_loss(logits23, train_y23)
    loss23.backward()
    optimizer23.step()
    loss_curve23.append(float(loss23.detach()))

model23.eval()
with torch.no_grad():
    train_prob23 = torch.sigmoid(model23(train_x23))
    final_loss23 = float(segmentation_loss(model23(train_x23), train_y23))
train_metrics23 = binary_overlap_metrics(train_prob23, train_y23, threshold=0.5)

assert final_loss23 < initial_loss23 * 0.25
assert train_metrics23["dice"] > 0.95
assert len(loss_curve23) == 100
assert all(math.isfinite(value) for value in loss_curve23)
print({"initial_loss": round(initial_loss23, 4), "final_loss": round(final_loss23, 4),
       "train": train_metrics23})


## 9. 阈值只在 validation 选择

sigmoid 概率的默认阈值 0.5 并非永远最优，特别是在类别不平衡、代价不对称或概率未校准时。下面只用 validation 的 mean Dice 从预先声明的候选集选择阈值，随后冻结并一次性应用到 test。若多个候选分数完全相同，本示例固定选择较小阈值；具体业务也可以预先约定更偏 precision 的规则，但不能看过 test 再改变。

真实项目应同时报告 PR 曲线、不同对象尺寸、空/非空、设备/地区/时间分群，并避免反复查看 test 后修改候选阈值。


In [ ]:
def select_threshold(validation_scores, candidates):
    if not candidates or set(validation_scores) != set(candidates):
        raise ValueError("候选阈值与验证分数字典必须非空且一一对应")
    if not all(math.isfinite(validation_scores[threshold]) for threshold in candidates):
        raise ValueError("验证分数必须为有限数")
    return max(candidates, key=lambda threshold: (validation_scores[threshold], -threshold))

# 确定性 oracle：最高分优先；完全同分时稳定选择较小阈值。
oracle_candidates23 = [0.25, 0.50, 0.75]
assert select_threshold({0.25: 0.7, 0.50: 0.9, 0.75: 0.8}, oracle_candidates23) == 0.50
assert select_threshold({0.25: 0.9, 0.50: 0.9, 0.75: 0.8}, oracle_candidates23) == 0.25

model23.eval()
with torch.no_grad():
    val_prob23 = torch.sigmoid(model23(val_x23))

threshold_candidates23 = [0.25, 0.35, 0.45, 0.50, 0.60, 0.70, 0.80]
validation_scores23 = {
    threshold: binary_overlap_metrics(val_prob23, val_y23, threshold=threshold)["dice"]
    for threshold in threshold_candidates23
}
selected_threshold23 = select_threshold(validation_scores23, threshold_candidates23)
# 阈值冻结之后才计算 test 预测和指标；test_y23 从未参与上面的选择。
with torch.no_grad():
    test_prob23 = torch.sigmoid(model23(test_x23))
test_metrics23 = binary_overlap_metrics(test_prob23, test_y23,
                                        threshold=selected_threshold23)

assert selected_threshold23 == select_threshold(validation_scores23, threshold_candidates23)
assert all(math.isfinite(score) for score in validation_scores23.values())
assert 0.0 <= test_metrics23["iou"] <= 1.0
assert 0.0 <= test_metrics23["dice"] <= 1.0
assert test_metrics23["evaluated"] == len(test_y23)
print({"validation": validation_scores23, "selected": selected_threshold23,
       "test": test_metrics23})


## 10. 失败反例：未对齐就 concat，以及概率/logit 混用

奇数尺寸在第二次上采样后少一行一列，直接 `torch.cat` 会报 shape 错误。另一个常见静默错误是先 sigmoid，再把概率传给 `binary_cross_entropy_with_logits`：该函数会把概率再次当 logit 做 sigmoid，损失和梯度都被改变。

若概率阈值为 $t$，等价 logit 阈值是 $\log(t/(1-t))$；只有概率阈值 0.5 对应 logit 阈值 0。


In [ ]:
decoder_bad = torch.zeros(1, 4, 30, 34)
skip_odd = torch.zeros(1, 4, 31, 35)
concat_error23 = None
try:
    _ = torch.cat([decoder_bad, skip_odd], dim=1)
except RuntimeError as exc:
    concat_error23 = str(exc)

sample_logit = torch.tensor([2.0], requires_grad=True)
sample_target = torch.tensor([1.0])
correct_bce = F.binary_cross_entropy_with_logits(sample_logit, sample_target)
wrong_double_sigmoid = F.binary_cross_entropy_with_logits(torch.sigmoid(sample_logit), sample_target)
equivalent_logit_threshold = math.log(selected_threshold23 / (1 - selected_threshold23))

assert concat_error23 is not None
assert align_to(decoder_bad, skip_odd).shape[-2:] == (31, 35)
assert not torch.isclose(correct_bce, wrong_double_sigmoid)
assert ((test_prob23 >= selected_threshold23) ==
        (torch.logit(test_prob23.clamp(1e-6, 1-1e-6)) >= equivalent_logit_threshold)).all()
print("预期 concat 失败:", concat_error23.split("\n")[0])


## 11. 全分辨率 tile：覆盖、halo 与边界效应

超大图通常不能一次放入显存。重叠 tiling 至少要保证：最后一个 tile 贴住右/下边界、每个像素覆盖次数大于零、重叠区用权重融合。直接平均 logits 是一种基线；对概率平均、中心裁剪或 Hann 权重会得到不同结果，必须版本化。

卷积依赖邻域，tile 边缘缺少整图上下文，因此即使重叠平均也不保证与整图推理完全相同。生产常增加 halo：读取更大的输入块，只保留中心可信区域；还要统一 padding mode、缩放尺度和坐标回写。


In [ ]:
def axis_starts(length, tile, overlap):
    if not (0 <= overlap < tile <= length):
        raise ValueError("要求 0 <= overlap < tile <= length")
    starts = list(range(0, length - tile + 1, tile - overlap))
    if starts[-1] != length - tile:
        starts.append(length - tile)
    return starts

def tile_slices(height, width, tile=24, overlap=8):
    return [(slice(y, y + tile), slice(x, x + tile))
            for y in axis_starts(height, tile, overlap)
            for x in axis_starts(width, tile, overlap)]

full_image23, _ = make_segmentation_data(1, 47, 53, SEED + 20)
accumulator = torch.zeros_like(full_image23)
coverage = torch.zeros_like(full_image23)
tiles23 = tile_slices(47, 53, tile=24, overlap=8)
for ys, xs in tiles23:
    accumulator[..., ys, xs] += full_image23[..., ys, xs]
    coverage[..., ys, xs] += 1
reconstructed23 = accumulator / coverage

with torch.no_grad():
    full_logits23 = model23(full_image23)
    tiled_logits23 = torch.zeros_like(full_logits23)
    tile_weights23 = torch.zeros_like(full_logits23)
    for ys, xs in tiles23:
        tiled_logits23[..., ys, xs] += model23(full_image23[..., ys, xs])
        tile_weights23[..., ys, xs] += 1
    tiled_logits23 /= tile_weights23
seam_gap23 = float((full_logits23 - tiled_logits23).abs().mean())

assert float(coverage.min()) >= 1
assert torch.allclose(reconstructed23, full_image23)
assert tiles23[-1][0].stop == 47 and tiles23[-1][1].stop == 53
assert torch.isfinite(tiled_logits23).all()
print({"tiles": len(tiles23), "max_coverage": int(coverage.max()),
       "whole_vs_tiled_mean_abs_gap": round(seam_gap23, 6)})


## 12. 梯度、模型指纹与发布清单

分割制品必须绑定输入通道/色彩空间、归一化、输出类别次序、阈值、空 mask policy、resize/tiling 版本和权重。只保存 `.pt` 权重会让服务端无法判断 0.5 是 probability 阈值还是 logit 阈值。

下面再做一次反向传播，验证编码器、解码器和输出头都有有限非零梯度；随后对排序后的 `state_dict` 计算 SHA-256。


In [ ]:
model23.train()
optimizer23.zero_grad(set_to_none=True)
probe_loss23 = segmentation_loss(model23(train_x23[:4]), train_y23[:4])
probe_loss23.backward()
gradients23 = {name: float(parameter.grad.norm()) for name, parameter in model23.named_parameters()
               if parameter.grad is not None}

def state_fingerprint23(model):
    digest = sha256()
    for name, tensor in sorted(model.state_dict().items()):
        value = tensor.detach().cpu().contiguous()
        digest.update(name.encode("utf-8"))
        digest.update(str(value.dtype).encode("ascii"))
        digest.update(str(tuple(value.shape)).encode("ascii"))
        digest.update(value.numpy().tobytes())
    return digest.hexdigest()

fingerprint23 = state_fingerprint23(model23)
manifest23 = {
    "artifact": "unet-base4-binary-toy-v1",
    "architecture": {"base": 4, "down_levels": 2, "output_channels": 1},
    "input": {"layout": "NCHW", "channels": 1, "range": [0.0, 1.0]},
    "output": {"kind": "pixel_logits", "threshold_space": "probability",
               "threshold": selected_threshold23, "empty_policy": "perfect"},
    "tiling": {"tile": 24, "overlap": 8, "fusion": "uniform_logit_average_demo"},
    "seed": SEED, "torch": torch.__version__, "state_dict_sha256": fingerprint23,
}

assert gradients23["inc.net.0.weight"] > 0
assert gradients23["up1.up.weight"] > 0
assert gradients23["out_conv.weight"] > 0
assert all(math.isfinite(value) for value in gradients23.values())
assert len(fingerprint23) == 64
assert state_fingerprint23(model23) == fingerprint23
print(json.dumps(manifest23, ensure_ascii=False, indent=2))


## 13. 生产边界与资料

真实分割还要解决：按患者/地块/视频来源切分防泄漏；标注版本和 annotator 一致性；类别长尾与小目标采样；多尺度增强但保持 mask 插值为 nearest；连通域/孔洞等后处理；边界指标 Hausdorff/Boundary IoU；推理显存和 tile halo；跨设备/地域漂移；阈值校准、人工复核与回滚。

若任务是多类互斥分割，应改为 `[N,C,H,W]` logits + cross-entropy，并用 argmax；若是多标签像素任务，则每通道独立 sigmoid 和阈值。两者不可仅靠改 `out_channels` 混用。

原始论文与官方资料：

- Ronneberger、Fischer、Brox，[U-Net: Convolutional Networks for Biomedical Image Segmentation](https://arxiv.org/abs/1505.04597)，2015。
- Milletari 等，[V-Net: Fully Convolutional Neural Networks for Volumetric Medical Image Segmentation](https://arxiv.org/abs/1606.04797)，Dice loss 的相关来源。
- PyTorch 官方文档：[BCEWithLogitsLoss](https://pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html)、[ConvTranspose2d](https://pytorch.org/docs/stable/generated/torch.nn.ConvTranspose2d.html)、[`interpolate`](https://pytorch.org/docs/stable/generated/torch.nn.functional.interpolate.html)。

结论边界：这里只证明手写 U-Net 在受控奇数尺寸合成图上能执行、反传并过拟合；没有证明真实分割质量、跨域鲁棒性或临床/安全可用性。
